In [5]:
import csv
import math
from pathlib import Path


def csv_to_latex_table(
    csv_path: str | Path,
    caption: str = "",
    label: str = "",
    decimal_places: int = 4,
    highlight_best: bool = True,
    best_is: str = "min",          # "min" | "max"
    col_rename: dict[str, str] | None = None,
    row_rename: dict[str, str] | None = None,
) -> str:
    """
    Convert a CSV file of numeric results to a LaTeX table.

    Parameters
    ----------
    csv_path : path-like
        Path to the CSV file. The first column is treated as a row-label
        column; all remaining columns must be numeric.
    caption : str
        Text for \\caption{}.  Omitted if empty.
    label : str
        Text for \\label{}.  Omitted if empty.
    decimal_places : int
        Number of decimal places passed to Python's round() before
        LaTeX receives the value.  LaTeX itself does no additional
        rounding.
    highlight_best : bool
        If True, the best value in each numeric column is typeset in
        bold via \\textbf{}.
    best_is : "min" | "max"
        Which extremum counts as "best".
    col_rename : dict[str, str] | None
        Optional mapping {raw_header -> display_header}.
    row_rename : dict[str, str] | None
        Optional mapping {raw_row_label -> display_row_label}.

    Returns
    -------
    str
        A complete LaTeX table environment as a string.
    """
    csv_path = Path(csv_path)
    col_rename = col_rename or {}
    row_rename = row_rename or {}

    # ── 1. Parse CSV ────────────────────────────────────────────────────────
    with csv_path.open(newline="") as fh:
        reader = csv.DictReader(fh)
        fieldnames = reader.fieldnames or []
        rows = list(reader)

    if not fieldnames:
        raise ValueError(f"No header found in {csv_path}")

    row_col, *num_cols = fieldnames

    # Convert to floats
    data: list[dict[str, float | str]] = []
    for row in rows:
        entry: dict[str, float | str] = {row_col: row[row_col]}
        for col in num_cols:
            entry[col] = float(row[col])
        data.append(entry)

    # ── 2. Find best value per numeric column ───────────────────────────────
    best: dict[str, float] = {}
    if highlight_best:
        selector = min if best_is == "min" else max
        for col in num_cols:
            best[col] = selector(float(r[col]) for r in data)  # type: ignore[arg-type]

    # ── 3. Format a single numeric cell ─────────────────────────────────────
    def fmt(value: float, col: str) -> str:
        rounded = round(value, decimal_places)
        text = f"{rounded:.{decimal_places}f}"
        if highlight_best and math.isclose(value, best[col], rel_tol=1e-9):
            return rf"\textbf{{{text}}}"
        return text

    # ── 4. Build header row ──────────────────────────────────────────────────
    def display_col(c: str) -> str:
        return col_rename.get(c, c).replace("_", r"\_")

    header_cells = [display_col(row_col)] + [display_col(c) for c in num_cols]
    header_line = " & ".join(header_cells) + r" \\"

    # ── 5. Build data rows ───────────────────────────────────────────────────
    body_lines: list[str] = []
    for entry in data:
        row_label = row_rename.get(str(entry[row_col]), str(entry[row_col])).replace("_", " ")
        cells = [row_label] + [fmt(float(entry[c]), c) for c in num_cols]  # type: ignore[arg-type]
        body_lines.append(" & ".join(cells) + r" \\")

    # ── 6. Column spec: l for label, c for each numeric column ──────────────
    col_spec = "l" + "c" * len(num_cols)

    # ── 7. Assemble table environment ───────────────────────────────────────
    lines: list[str] = [
        r"\begin{table}[tb]",
        r"  \centering",
    ]
    if caption:
        lines.append(rf"  \caption{{{caption}}}")
    if label:
        lines.append(rf"  \label{{{label}}}")
    lines += [
        r"   \resizebox{\columnwidth}{!}{%",
        rf"  \begin{{tabular}}{{{col_spec}}}",
        r"    \toprule",
        f"    {header_line}",
        r"    \midrule",
    ]
    for bl in body_lines:
        lines.append(f"    {bl}")
    lines += [
        r"    \bottomrule",
        r"  \end{tabular}",
        r"   }",
        r"\end{table}",
    ]

    return "\n".join(lines)


# ── Convenience: batch-convert a list of CSV files ──────────────────────────
def csvs_to_latex(
    csv_paths: list[str | Path],
    output_path: str | Path | None = None,
    **kwargs,
) -> str:
    """
    Convert multiple CSV files to LaTeX tables and concatenate them.

    Parameters
    ----------
    csv_paths : list of path-like
        Input CSV files.  Each becomes one table.
    output_path : path-like or None
        If given, the combined output is written here.
    **kwargs
        Forwarded to csv_to_latex_table for every file.

    Returns
    -------
    str
        All tables joined by blank lines.
    """
    tables = []
    for path in csv_paths:
        path = Path(path)
        kw = dict(kwargs)
        kw.setdefault("caption", path.stem.replace("_", " ").title())
        kw.setdefault("label", f"tab:{path.stem}")
        tables.append(csv_to_latex_table(path, **kw))

    combined = "\n\n".join(tables)

    if output_path is not None:
        Path(output_path).write_text(combined)

    return combined

In [6]:
latex = csv_to_latex_table(
    #"stats_mse_by_distribution.csv",
    "stats_mse_by_payoff.csv",
    #"stats_errors.csv",
    caption="MSE comparison across distributions",
    label="tab:mse_comparison",
    decimal_places=4,
    highlight_best=False,
    best_is="min",
    col_rename={
        "dist_name":              "Distribution",
        "gpq_mse":               "GPQ",
        "hsgp_mse":              "HSGP",
        "quantum_analytical_mse": "QA-HSGP (analytical)",
        "quantum_mse":           "QA-HSGP",
    },
)
print(latex)

\begin{table}[tb]
  \centering
  \caption{MSE comparison across distributions}
  \label{tab:mse_comparison}
   \resizebox{\columnwidth}{!}{%
  \begin{tabular}{lcccc}
    \toprule
    payoff\_name & GPQ & HSGP & QA-HSGP (analytical) & QA-HSGP \\
    \midrule
    deductible with limit & 0.0160 & 0.0171 & 0.0171 & 0.0133 \\
    franchise deductible & 0.6271 & 1.8256 & 1.8256 & 0.1932 \\
    ordinary deductible & 0.0231 & 0.0239 & 0.0239 & 0.0143 \\
    policy limit & 0.5672 & 1.7747 & 1.7747 & 0.1939 \\
    stop loss & 0.0231 & 0.0239 & 0.0239 & 0.0143 \\
    \bottomrule
  \end{tabular}
   }
\end{table}


## QPE spectral truncation diagnostic

The circuit-based quantum method gives suspiciously good MSE because the QPE with finite precision $\tau$ bits accidentally applies **implicit spectral truncation**.

Any eigenvalue $\lambda_r < F^2 / 2^\tau$ (where $F^2 = \|X\|_F^2$) gets rounded to QPE bin $j=0$ (phase $\theta=0$) and receives **zero weight** in the mean estimate, since `conditional_rotation_mean` skips `j=0` by default (`include_zero_bin=False`). The classical HSGP uses all eigenvalues. For ill-conditioned cases like Pareto the small eigenvalues add noise; truncating them accidentally regularises.

The cell below reproduces the effect analytically, showing which eigenvalues get truncated and how the truncated inversion compares to the full one.

In [7]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import numpy as np
import json
from gaussian_quantum.hilbert_space_approx import hilbert_space_features
from gaussian_quantum.qpca import _bit_reverse

# ── Reproduce one stored experiment (pareto × franchise_deductible) ──────────
with open("results_raw.json") as f:
    results = json.load(f)

# Pick the worst-looking case for classical: pareto/franchise_deductible
r = next(x for x in results if x["dist_name"] == "pareto"
         and x["payoff_name"] == "franchise_deductible")

print(f"Case: {r['dist_name']} × {r['payoff_name']}")
print(f"  exact          = {r['exact']:.6f}")
print(f"  hsgp_mean      = {r['hsgp_mean']:.6f}  (bias {r['hsgp_mean']-r['exact']:+.3f})")
print(f"  quantum_mean   = {r['quantum_mean']:.6f}  (bias {r['quantum_mean']-r['exact']:+.3f})")
print(f"  N={r['N']}, M={r['M']}, noise_std={r['noise_std']}, tau={r['n_eigenvalue_qubits']}")

# ── Rebuild the feature matrix (need the same training data) ─────────────────
from gaussian_quantum.insurance import make_integrand, PAYOFF_DEFAULTS, DISTRIBUTIONS
from experiments import EXPERIMENT_DIST_PARAMS, _sample_points

dist_params = EXPERIMENT_DIST_PARAMS["pareto"]
integrand, domain, dist = make_integrand("pareto", dist_params, "franchise_deductible",
                                          PAYOFF_DEFAULTS["franchise_deductible"])
a, b = domain
midpoint = (a + b) / 2.0
rng = np.random.default_rng(r.get("seed", 679))
X_raw = _sample_points(dist, a, b, r["N"], r["point_strategy"])
y_eval = np.squeeze(integrand(X_raw)) + rng.normal(0.0, r["noise_std"], size=r["N"])
X_eval = (X_raw - midpoint).reshape(-1, 1)

X_feat, _, _, _ = hilbert_space_features(X_eval, r["M"], r["L"],
                                          r["length_scale"], r.get("amplitude", 1.0))

# ── Eigenvalue spectrum of XᵀX ──────────────────────────────────────────────
XtX = X_feat.T @ X_feat
F_sq = float(np.trace(XtX))
eigenvalues = np.linalg.eigvalsh(XtX)
eigenvalues = np.maximum(eigenvalues, 0.0)

tau = r["n_eigenvalue_qubits"]        # 8 from stored results
truncation_threshold = F_sq / (2 ** tau)
truncated = eigenvalues[eigenvalues < truncation_threshold]
kept       = eigenvalues[eigenvalues >= truncation_threshold]

print(f"\n── Eigenvalue spectrum ──────────────────────────────────────────")
print(f"  F² = tr(XᵀX)          = {F_sq:.4f}")
print(f"  QPE bits τ             = {tau}")
print(f"  Truncation threshold   = F²/2^τ = {truncation_threshold:.6f}")
print(f"  Eigenvalues KEPT  ({len(kept):2d}) : min={kept.min():.4f}, max={kept.max():.4f}")
print(f"  Eigenvalues DROPPED ({len(truncated):2d}) : {np.sort(truncated)[:5].round(6).tolist()} ...")

# ── Compare full vs truncated matrix inversion ───────────────────────────────
noise_var = r["noise_std"] ** 2
z_mu_feat, _, _, _ = hilbert_space_features(
    np.array([[midpoint]]),   # placeholder; kernel_mean_features would be better
    r["M"], r["L"], r["length_scale"], r.get("amplitude", 1.0)
)

from gaussian_quantum.hilbert_space_approx import kernel_mean_features
z_mu, _, _, _ = kernel_mean_features(
    (r["centered_domain"][0], r["centered_domain"][1]),
    r["M"], r["L"], r["length_scale"], r.get("amplitude", 1.0)
)

Xty = X_feat.T @ y_eval

# Full inversion (= classical HSGP)
A_full = XtX + noise_var * np.eye(r["M"])
mean_full = z_mu @ np.linalg.solve(A_full, Xty)

# Truncated inversion (mimics QPE: zero out eigenvalues below threshold)
eigenvalues_t, eigenvectors_t = np.linalg.eigh(XtX)
eigenvalues_t = np.maximum(eigenvalues_t, 0.0)
mask = eigenvalues_t >= truncation_threshold
print(f"\n── Inversion comparison ─────────────────────────────────────────")
print(f"  Full inversion mean     (= HSGP)       = {mean_full:.6f}")

for tau_test in [4, 6, 8, 10, 12, 100]:
    thr = F_sq / (2 ** tau_test)
    m = eigenvalues_t >= thr
    n_kept = m.sum()
    # Reconstruct truncated (XᵀX + σ²I)⁻¹ in eigenbasis
    inv_diag = np.where(m, 1.0 / (eigenvalues_t + noise_var), 0.0)
    A_inv_trunc = eigenvectors_t @ np.diag(inv_diag) @ eigenvectors_t.T
    mean_trunc = z_mu @ A_inv_trunc @ Xty
    marker = " ← stored results" if tau_test == tau else ""
    print(f"  τ={tau_test:3d}  threshold={thr:.4f}  kept {n_kept}/{r['M']} eigs  "
          f"mean={mean_trunc:.6f}  bias={mean_trunc - r['exact']:+.3f}{marker}")

print(f"\n  Exact answer = {r['exact']:.6f}")


Case: pareto × franchise_deductible
  exact          = 1.485000
  hsgp_mean      = 4.353202  (bias +2.868)
  quantum_mean   = 1.032486  (bias -0.453)
  N=16, M=16, noise_std=0.01, tau=8

── Eigenvalue spectrum ──────────────────────────────────────────
  F² = tr(XᵀX)          = 16.2448
  QPE bits τ             = 8
  Truncation threshold   = F²/2^τ = 0.063456
  Eigenvalues KEPT  ( 6) : min=0.3497, max=11.2537
  Eigenvalues DROPPED (10) : [0.0, 0.0, 0.0, 0.0, 0.0] ...

── Inversion comparison ─────────────────────────────────────────
  Full inversion mean     (= HSGP)       = 4.353202
  τ=  4  threshold=1.0153  kept 3/16 eigs  mean=0.450493  bias=-1.035
  τ=  6  threshold=0.2538  kept 6/16 eigs  mean=2.092833  bias=+0.608
  τ=  8  threshold=0.0635  kept 6/16 eigs  mean=2.092833  bias=+0.608 ← stored results
  τ= 10  threshold=0.0159  kept 7/16 eigs  mean=0.111445  bias=-1.374
  τ= 12  threshold=0.0040  kept 7/16 eigs  mean=0.111445  bias=-1.374
  τ=100  threshold=0.0000  kept 13/16 eigs 

In [9]:
import sys, os; sys.path.insert(0, os.path.dirname(os.getcwd()))
import numpy as np, json
from gaussian_quantum.hilbert_space_approx import hilbert_space_features, kernel_mean_features
from gaussian_quantum.qpca import _bit_reverse
from experiments import EXPERIMENT_DIST_PARAMS, _sample_points
from gaussian_quantum.insurance import make_integrand, PAYOFF_DEFAULTS

# ── Reproduce pareto × franchise_deductible (most extreme case) ──────────────
with open("results_raw.json") as f:
    results = json.load(f)
r = next(x for x in results if x["dist_name"] == "pareto"
         and x["payoff_name"] == "franchise_deductible")

integrand, domain, dist = make_integrand(
    "pareto", EXPERIMENT_DIST_PARAMS["pareto"],
    "franchise_deductible", PAYOFF_DEFAULTS["franchise_deductible"])
a, b = domain; midpoint = (a + b) / 2.0
X_raw = _sample_points(dist, a, b, r["N"], r["point_strategy"])
y_eval = np.squeeze(integrand(X_raw)) + np.random.default_rng(679).normal(0.0, r["noise_std"], size=r["N"])
X_eval = (X_raw - midpoint).reshape(-1, 1)
X_feat, _, _, _ = hilbert_space_features(X_eval, r["M"], r["L"], r["length_scale"], r["amplitude"])
XtX = X_feat.T @ X_feat; F_sq = float(np.trace(XtX))
eigenvalues_t, eigenvectors_t = np.linalg.eigh(XtX)
eigenvalues_t = np.maximum(eigenvalues_t, 0.0)
noise_var = r["noise_std"] ** 2
tau = r["n_eigenvalue_qubits"]
z_mu, _, _, _ = kernel_mean_features(
    (r["centered_domain"][0], r["centered_domain"][1]),
    r["M"], r["L"], r["length_scale"], r["amplitude"])
Xty = X_feat.T @ y_eval

# ── QPE bin assignment for each eigenvalue ───────────────────────────────────
# QPE encodes eigenphase theta = lam/F² as register value j_actual = bit_reverse(round(theta·2^τ)).
# The conditional rotation for register value j_actual uses theta_tilde = bit_reverse(j_actual)/2^τ
# = round(theta·2^τ)/2^τ, giving lam_tilde = round(theta·2^τ)/2^τ · F².
# j_actual == 0 ↔ round(theta·2^τ) == 0 ↔ lam < F²/(2^(τ+1)) → eigenvalue DROPPED.

drop_threshold = F_sq / (2 ** (tau + 1))
print(f"Case: {r['dist_name']} × {r['payoff_name']}")
print(f"  N={r['N']}, M={r['M']}, noise_std={r['noise_std']}, tau={tau}")
print(f"  F² = {F_sq:.4f},   drop threshold = F²/2^(τ+1) = {drop_threshold:.5f}")
print()
print(f"{'lambda':>10s}  {'weight 1/(λ+σ²)':>16s}  {'round(θ·2^τ)':>13s}  status")
print("-" * 68)
for lam in np.sort(eigenvalues_t):
    j_rev = int(round(lam / F_sq * (2 ** tau))) % (2 ** tau)
    weight = 1.0 / (lam + noise_var)
    status = "DROPPED by QPE" if j_rev == 0 else "kept"
    print(f"  {lam:10.5f}  {weight:>16.1f}  {j_rev:>13d}  {status}")

# ── Show what each method effectively computes ────────────────────────────────
# Classical: all eigenvalues
A_full = XtX + noise_var * np.eye(r["M"])
mean_full = float(z_mu @ np.linalg.solve(A_full, Xty))

# QPE truncated: drop eigenvalues rounded to bin 0
inv_diag_qpe = np.array([
    0.0 if int(round(lam / F_sq * 2**tau)) % 2**tau == 0
    else 1.0 / (int(round(lam / F_sq * 2**tau)) / 2**tau * F_sq + noise_var)
    for lam in eigenvalues_t
])
mean_qpe = float(z_mu @ eigenvectors_t @ np.diag(inv_diag_qpe) @ eigenvectors_t.T @ Xty)

n_dropped = sum(1 for lam in eigenvalues_t
                if int(round(lam / F_sq * 2**tau)) % 2**tau == 0)
print()
print(f"  Classical HSGP (all {r['M']} modes):               mean = {mean_full:.4f}  (stored: {r['hsgp_mean']:.4f})")
print(f"  QPE nearest-bin ({r['M']-n_dropped} modes kept, {n_dropped} dropped): mean = {mean_qpe:.4f}")
print(f"  Actual circuit (QPE + leakage + shots):        mean = {r['quantum_mean']:.4f}")
print(f"  Exact answer:                                       = {r['exact']:.4f}")
print()
print("ROOT CAUSE:")
print(f"  The {n_dropped} dropped eigenvalues (all ≤ {max(lam for lam in eigenvalues_t if int(round(lam/F_sq*2**tau))%2**tau==0):.5f})")
print(f"  each get weight 1/(λ+σ²) ≥ {1.0/(max(lam for lam in eigenvalues_t if int(round(lam/F_sq*2**tau))%2**tau==0)+noise_var):.0f} in the classical solve.")
print(f"  Even tiny v_r^T X^T y ≠ 0 (numerical noise) gets amplified enormously.")
print(f"  QPE with τ={tau} cannot distinguish these from zero → they are silently dropped,")
print(f"  which accidentally regularises the inversion and gives a better-looking mean.")
print(f"  This is NOT a quantum advantage — it is implicit regularisation via QPE precision.")


Case: pareto × franchise_deductible
  N=16, M=16, noise_std=0.01, tau=8
  F² = 16.2448,   drop threshold = F²/2^(τ+1) = 0.03173

    lambda   weight 1/(λ+σ²)   round(θ·2^τ)  status
--------------------------------------------------------------------
     0.00000           10000.0              0  DROPPED by QPE
     0.00000           10000.0              0  DROPPED by QPE
     0.00000           10000.0              0  DROPPED by QPE
     0.00000           10000.0              0  DROPPED by QPE
     0.00000           10000.0              0  DROPPED by QPE
     0.00000           10000.0              0  DROPPED by QPE
     0.00000            9999.8              0  DROPPED by QPE
     0.00000            9812.4              0  DROPPED by QPE
     0.00038            2067.0              0  DROPPED by QPE
     0.02305              43.2              0  DROPPED by QPE
     0.34966               2.9              6  kept
     0.82075               1.2             13  kept
     1.00224              